# Tradução em teste para modelos Transformer v80+

Notebook adaptado ao fluxo real do treinamento:
- os textos não vêm em colunas `en`/`pt`
- as frases reais são lidas por coordenadas (`block_id`, `src_offset`, `tgt_offset`, `src_length`, `tgt_length`)
- o parquet apenas aponta para os pedaços corretos dentro dos blocos `.bin`

Isso replica exatamente a lógica usada em `training-dataset-maior-2.py`.

In [10]:
import json
import os
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import tokenizers
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer

from models.transformer import Transformer

print("CUDA disponível:", torch.cuda.is_available())

CUDA disponível: True


In [2]:
TOKENIZED_DIR = "/media/alvarinho/dados/Datasets/refined/traducao/tokenized"
METADATA_PATH = os.path.join(TOKENIZED_DIR, "analise_textos_tokenized.parquet")
BLOCKS_DIR = os.path.join(TOKENIZED_DIR, "blocks")
PATH_MODELS = "artifacts"
PATH_CONFIGS = "configs"


def resolve_tokenizer_path():
    candidates = [
        "artifacts/tokenizer_en_pt_es_60000.json",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("Nenhum tokenizer compatível encontrado em artifacts/")


def load_metadata():
    required = {
        "global_row", "dataset_id", "index", "split", "block_id",
        "row_in_block", "src_offset", "tgt_offset", "src_length", "tgt_length",
    }
    df = pd.read_parquet(METADATA_PATH)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Parquet incompatível: colunas faltando {sorted(missing)}")
    df["split"] = df["split"].astype(str).str.lower()
    return df


def read_tokens_from_metadata_row(row: pd.Series) -> Tuple[List[int], List[int]]:
    block_id = int(row["block_id"])
    src_offset = int(row["src_offset"])
    tgt_offset = int(row["tgt_offset"])
    src_length = int(row["src_length"])
    tgt_length = int(row["tgt_length"])

    src_path = os.path.join(BLOCKS_DIR, f"block_{block_id:05d}_src.bin")
    tgt_path = os.path.join(BLOCKS_DIR, f"block_{block_id:05d}_tgt.bin")

    src_mem = np.memmap(src_path, dtype=np.int32, mode="r")
    tgt_mem = np.memmap(tgt_path, dtype=np.int32, mode="r")

    src_ids = src_mem[src_offset:src_offset + src_length].astype(np.int64).tolist()
    tgt_ids = tgt_mem[tgt_offset:tgt_offset + tgt_length].astype(np.int64).tolist()
    return src_ids, tgt_ids


def normalize_prediction(prediction):
    if isinstance(prediction, torch.Tensor):
        prediction = prediction.detach().cpu().tolist()
    elif isinstance(prediction, np.ndarray):
        prediction = prediction.tolist()

    while isinstance(prediction, (list, tuple)) and len(prediction) == 1 and isinstance(prediction[0], (list, tuple, np.ndarray, torch.Tensor)):
        prediction = prediction[0]

    if isinstance(prediction, torch.Tensor):
        prediction = prediction.detach().cpu().tolist()
    elif isinstance(prediction, np.ndarray):
        prediction = prediction.tolist()

    return [int(token) for token in prediction]


metadata_df = load_metadata()
print("Metadata carregado:", metadata_df.shape)
print(metadata_df.head(3).to_string(index=False))


tokenizer_path = resolve_tokenizer_path()
tokenizer = tokenizers.Tokenizer.from_file(tokenizer_path)
print(f"Tokenizer carregado: {tokenizer_path}")

Metadata carregado: (26092729, 10)
 global_row dataset_id  index split  block_id  row_in_block  src_offset  tgt_offset  src_length  tgt_length
          0 EMEA_en_es      2  test         0             0           0           0          18          17
          1 EMEA_en_es      4 train         0             1          18          17           7          10
          2 EMEA_en_es      5  test         0             2          25          27          16          18
Tokenizer carregado: artifacts/tokenizer_en_pt_es_60000.json


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {}
for file in sorted(os.listdir(PATH_MODELS)):
    if not file.endswith(".pt") or not file.startswith("model_"):
        continue

    model_name = file.split(".")[0].replace("model_", "")
    try:
        version = int(model_name.lstrip("v")) if model_name.startswith("v") else None
    except ValueError:
        version = None

    if version is None or version < 84:
        continue

    config_path = os.path.join(PATH_CONFIGS, f"{model_name}_config.json")
    if not os.path.exists(config_path):
        print(f"Ignorando {model_name}: config não encontrado em {config_path}")
        continue

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    model = Transformer(
        embedding_dim=config["embedding_dim"],
        encoder_num_layers=config["encoder_num_layers"],
        decoder_num_layers=config["decoder_num_layers"],
        encoder_hidden_dim=config["encoder_hidden_dim"],
        decoder_hidden_dim=config["decoder_hidden_dim"],
        encoder_num_heads=config["encoder_num_heads"],
        decoder_num_heads=config["decoder_num_heads"],
        encoder_dropout=config["encoder_dropout"],
        decoder_dropout=config["decoder_dropout"],
        vocab_size=config.get("vocab_size", 120000),
        kv_cache=True,
        pad_idx=0,
        separate_embedding=config.get("separate_embedding", False),
    )

    state = torch.load(os.path.join(PATH_MODELS, file), map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    models[model_name] = model

print(f"Modelos carregados: {list(models.keys())}")

Modelos carregados: ['v84', 'v85', 'v86']


In [6]:
# Seleciona 10 exemplos reais, 5 em português e 5 em espanhol, preferindo o split de validação
sample_size_per_lang = 5

selected_rows = []
for lang, suffix in [("pt", "_en_pt"), ("es", "_en_es")]:
    candidates = metadata_df[
        (metadata_df["split"] == "val") &
        metadata_df["dataset_id"].str.endswith(suffix)
    ].copy()

    if len(candidates) < sample_size_per_lang:
        fallback = metadata_df[
            (metadata_df["split"] == "test") &
            metadata_df["dataset_id"].str.endswith(suffix)
        ].copy()
        candidates = pd.concat([candidates, fallback], ignore_index=True)

    selected_rows.append(candidates.head(sample_size_per_lang))

examples_df = pd.concat(selected_rows, ignore_index=True)
print("Exemplos escolhidos:")
print(examples_df[["global_row", "dataset_id", "split", "block_id", "src_offset", "tgt_offset", "src_length", "tgt_length"]].to_string(index=False))
print(f"\nTotal selecionado: {len(examples_df)} linhas")

Exemplos escolhidos:
 global_row dataset_id split  block_id  src_offset  tgt_offset  src_length  tgt_length
     618634 EMEA_en_pt   val         3      126023      148742          52          48
     618650 EMEA_en_pt   val         3      126312      149064          14          17
     618761 EMEA_en_pt   val         3      128860      152055          26          34
     618916 EMEA_en_pt   val         3      132961      156812           6           7
     618971 EMEA_en_pt   val         3      133861      157871          37          46
        386 EMEA_en_es   val         0        9267       10516          51          51
        407 EMEA_en_es   val         0        9965       11320          30          31
        439 EMEA_en_es   val         0       10905       12345          40          45
        518 EMEA_en_es   val         0       12869       14615          40          45
        565 EMEA_en_es   val         0       13905       15777          21          25

Total selecionado: 10

In [11]:
# Avalia os modelos em 10 exemplos reais escolhidos e calcula BLEU/ROUGE

def compute_rouge_f1(prediction: str, reference: str):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(reference, prediction)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }

model_metrics = []

for model_name, model in models.items():
    print(f"\n=== Modelo: {model_name} ===")
    hyps = []
    refs = []
    rouge_acc = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    rouge_count = 0

    for idx, row in examples_df.iterrows():
        src_ids, tgt_ids = read_tokens_from_metadata_row(row)

        src_text = tokenizer.decode(src_ids, skip_special_tokens=False)
        tgt_text = tokenizer.decode(tgt_ids, skip_special_tokens=False)

        target_lang_token = "<2pt>" if row["dataset_id"].endswith("_en_pt") else "<2es>"
        lang_token_id = tokenizer.token_to_id(target_lang_token)
        input_decoder = torch.tensor([[5, lang_token_id]], dtype=torch.long, device=DEVICE)

        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            pred = model.predict(
                src_tensor,
                bos_token_id=5,
                eos_token_id=6,
                max_len=max(128, len(tgt_ids) + 20),
                reset_cache=True,
                input_decoder=input_decoder,
            )

        pred_ids = normalize_prediction(pred)
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=False)

        hyps.append(pred_text)
        refs.append(tgt_text)

        rouge_values = compute_rouge_f1(pred_text, tgt_text)
        for key in rouge_acc:
            rouge_acc[key] += rouge_values[key]
        rouge_count += 1

        print(f"global_row={row['global_row']} | dataset_id={row['dataset_id']}")
        print(f"Entrada:  {src_text[:200]}")
        print(f"Esperado: {tgt_text[:200]}")
        print(f"Saída:    {pred_text[:200]}\n")

    bleu = BLEU()
    bleu_score = bleu.corpus_score(hyps, [refs]).score

    model_metrics.append({
        "model_name": model_name,
        "bleu": bleu_score,
        "rouge1": rouge_acc["rouge1"] / max(1, rouge_count),
        "rouge2": rouge_acc["rouge2"] / max(1, rouge_count),
        "rougeL": rouge_acc["rougeL"] / max(1, rouge_count),
    })

metrics_df = pd.DataFrame(model_metrics).sort_values("bleu", ascending=False)
metrics_df


=== Modelo: v84 ===
global_row=618634 | dataset_id=EMEA_en_pt
Entrada:  <BOS>What Enviage looks like and contents of the pack Enviage 300 mg film coated tablets are light - red , biconvex , ovaloid tablets , imprinted “ IU ” on one side and “ NVR ” on the other side . <EO
Esperado: <BOS><2pt>Qual o aspecto de Enviage e conteúdo da embalagem Enviage 300 mg é um comprimido vermelho claro , biconvexo , ovaloide , com impressão “ IU ” numa face e “ NVR ” na outra face . <EOS>
Saída:    <BOS><2pt>O que o Enviage parece e conteúdo da embalagem Enviage Enviage 300 mg comprimidos revestidos de comprimidos são comprimidos são os “ luz de luz ” de luz ” de luz , de luz , de forma de cor d

global_row=618650 | dataset_id=EMEA_en_pt
Entrada:  <BOS>Epivir is a medicine containing the active substance lamivudine . <EOS>
Esperado: <BOS><2pt>O Epivir é um medicamento que contém a substância activa lamivudina . <EOS>
Saída:    <BOS><2pt>Epivir é um medicamento que contém a substância activa substância

,model_name,bleu,rouge1,rouge2,rougeL
1,v85,44.545476,0.628819,0.477532,0.609401
0,v84,42.921016,0.640363,0.467442,0.605361
2,v86,40.788583,0.637236,0.447616,0.601677


In [16]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 5000)
# Tabela de configuração dos modelos carregados
model_config_rows = []
for model_name in sorted(models.keys()):
    config_path = os.path.join(PATH_CONFIGS, f"{model_name}_config.json")
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    model_config_rows.append({
        "model_name": model_name,
        "embedding_dim": cfg.get("embedding_dim"),
        "encoder_hidden_dim": cfg.get("encoder_hidden_dim"),
        "decoder_hidden_dim": cfg.get("decoder_hidden_dim"),
        "encoder_num_layers": cfg.get("encoder_num_layers"),
        "decoder_num_layers": cfg.get("decoder_num_layers"),
        "encoder_num_heads": cfg.get("encoder_num_heads"),
        "decoder_num_heads": cfg.get("decoder_num_heads"),
        "dropout_enc": cfg.get("encoder_dropout"),
        "dropout_dec": cfg.get("decoder_dropout"),
        "learning_rate": cfg.get("learning_rate"),
        "weight_decay": cfg.get("weight_decay"),
        "label_smoothing": cfg.get("label_smoothing"),
        "vocab_size": cfg.get("vocab_size"),
        "desc": cfg.get("desc")
    })

model_config_df = pd.DataFrame(model_config_rows)
model_config_df

,model_name,embedding_dim,encoder_hidden_dim,decoder_hidden_dim,encoder_num_layers,decoder_num_layers,encoder_num_heads,decoder_num_heads,dropout_enc,dropout_dec,learning_rate,weight_decay,label_smoothing,vocab_size,desc
0,v84,768,768,768,4,4,8,8,0.1,0.1,0.0001,0.01,0.0,120000,Treinando com as camadas pre normalizadas e co...
1,v85,768,768,768,4,4,8,8,0.1,0.1,0.0001,0.01,0.0,120000,Treinando com as camadas pre normalizadas e co...
2,v86,768,768,768,4,4,8,8,0.1,0.1,0.0001,0.01,0.1,120000,Adicionando label smoothing


In [ ]:
# Teste rápido em uma frase manual
manual_sentence = "You've heard of the genie of the lamp?"
src_ids = tokenizer.encode(manual_sentence).ids
src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

print(f"Frase manual: {manual_sentence}\n")
for model_name, model in models.items():
    with torch.no_grad():
        pred = model.predict(
            src_tensor,
            bos_token_id=5,
            eos_token_id=6,
            max_len=300,
            reset_cache=True,
        )
    pred_ids = normalize_prediction(pred)
    pred_text = tokenizer.decode(pred_ids, skip_special_tokens=False)
    print(f"{model_name}: {pred_text}")